In [1]:
import numpy as np 
import pandas as pd

In [2]:
pd.set_option("display.max_columns", None)

In [3]:
ls

 Volume in drive C is Windows X-Lite
 Volume Serial Number is FAAD-6DE0

 Directory of C:\Users\Admin\sandbox

07/28/2026  11:29 AM    <DIR>          .
07/28/2026  11:05 AM    <DIR>          ..
07/27/2026  10:34 PM    <DIR>          .ipynb_checkpoints
07/28/2026  12:19 AM            30,721 Data_Quality_and_Cleaning.ipynb
07/28/2026  11:29 AM            37,866 Feature_Engineering.ipynb
07/26/2026  06:06 PM    <DIR>          Portfolio
07/19/2026  10:55 AM    <DIR>          project
07/12/2026  11:49 AM           168,739 project 1.ipynb
07/12/2026  07:02 PM           343,114 sales v1.ipynb
07/12/2026  11:49 AM           168,739 Untitled.ipynb
07/12/2026  03:15 PM               617 Untitled1.ipynb
07/27/2026  10:34 PM                72 Untitled2.ipynb
07/28/2026  11:18 AM               617 Untitled3.ipynb
07/09/2026  04:11 PM    <DIR>          venv
               8 File(s)        750,485 bytes
               6 Dir(s)  132,829,777,920 bytes free


In [4]:
cd C:\Users\Admin\sandbox\Portfolio\lost sales\retail store data

C:\Users\Admin\sandbox\Portfolio\lost sales\retail store data


In [5]:
df = pd.read_csv("cleaned_inventory.csv")

In [6]:
df.head()

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer


here in feature engineering we have to investigat :
- revenue generation
- inventory efficiency
- demand fulfillment
- dtock availability
- revenue leakage
- store performance

In [7]:
df.shape

(73100, 15)

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 73100 entries, 0 to 73099
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Date                73100 non-null  str    
 1   Store ID            73100 non-null  str    
 2   Product ID          73100 non-null  str    
 3   Category            73100 non-null  str    
 4   Region              73100 non-null  str    
 5   Inventory Level     73100 non-null  int64  
 6   Units Sold          73100 non-null  int64  
 7   Units Ordered       73100 non-null  int64  
 8   Demand Forecast     73100 non-null  float64
 9   Price               73100 non-null  float64
 10  Discount            73100 non-null  int64  
 11  Weather Condition   73100 non-null  str    
 12  Holiday/Promotion   73100 non-null  int64  
 13  Competitor Pricing  73100 non-null  float64
 14  Seasonality         73100 non-null  str    
dtypes: float64(3), int64(5), str(7)
memory usage: 8.4 MB


In [9]:
df["Date"] = pd.to_datetime(df["Date"])

In [10]:
df.describe()

,Date,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Holiday/Promotion,Competitor Pricing
count,73100,73100.000000,73100.000000,73100.000000,73100.000000,73100.000000,73100.000000,73100.000000,73100.000000
mean,2023-01-01 00:00:00,274.469877,136.464870,110.004473,141.528727,55.135108,10.009508,0.497305,55.146077
min,2022-01-01 00:00:00,50.000000,0.000000,20.000000,0.000000,10.000000,0.000000,0.000000,5.030000
25%,2022-07-02 00:00:00,162.000000,49.000000,65.000000,53.670000,32.650000,5.000000,0.000000,32.680000
50%,2023-01-01 00:00:00,273.000000,107.000000,110.000000,113.015000,55.050000,10.000000,0.000000,55.010000
75%,2023-07-03 00:00:00,387.000000,203.000000,155.000000,208.052500,77.860000,15.000000,1.000000,77.820000
max,2024-01-01 00:00:00,500.000000,499.000000,200.000000,518.550000,100.000000,20.000000,1.000000,104.940000
std,NaN,129.949514,108.919406,52.277448,109.209174,26.021945,7.083746,0.499996,26.191408


### feature engineering starts here


### Actual SP

In [11]:
# Actual selling price aftero discount applied
df["Final price"]= (df["Price"] * (1 - df["Discount"] / 100))

In [12]:
df[["Price", "Discount","Final price"]].head()

,Price,Discount,Final price
0,33.50,20,26.800
1,63.01,20,50.408
2,27.99,10,25.191
3,32.72,10,29.448
4,73.64,0,73.640


### Revenue

In [13]:
#so calculated revenue will be
df["Revenue"] = (df["Units Sold"] * df["Final price"])

In [14]:
round(df["Revenue"].sum(),2)

np.float64(494971374.95)

### Inventory untilization


In [15]:
# calculating leftover inventory after selling
df["Remaining inventory"] = (df["Inventory Level"] - df["Units Sold"])

In [16]:
df[["Inventory Level","Units Sold", "Remaining inventory"]].head()

,Inventory Level,Units Sold,Remaining inventory
0,231,127,104
1,204,150,54
2,102,65,37
3,469,61,408
4,166,14,152


In [17]:
df["Inventory utilization %"] = (
    df["Units Sold"] / df["Inventory Level"] * 100)

In [18]:
df[["Inventory utilization %"]].head() 

,Inventory utilization %
0,54.978355
1,73.529412
2,63.725490
3,13.006397
4,8.433735


### demand gap & Demand fullfullment rate

this can callcuated by comparing the forecast and the units actually sold

In [19]:
df["Demand gap"] = (df["Demand Forecast"] - df["Units Sold"])

In [20]:
df["Demand gap"].head()

0    8.47
1   -5.96
2    9.02
3    1.18
4   -4.74
Name: Demand gap, dtype: float64

In [21]:
df["Fulfillment rate (%)"] = (df["Units Sold"] / df["Demand Forecast"]) * 100

In [22]:
#one more thing the forecast can be zero causing mathematical error, so 
df["Fulfillment Rate (%)"] = np.where(
    df["Demand Forecast"] > 0,
    (df["Units Sold"] / df["Demand Forecast"]) * 100,
    np.nan
)

df[["Fulfillment Rate (%)"]].head()

,Fulfillment Rate (%)
0,93.747693
1,104.137740
2,87.814104
3,98.102284
4,151.187905


### Stockout

meaning customer has demand but inventory isnt fullfilling it

In [23]:
df["Stockout"] = np.where(
    df["Demand Forecast"] > df["Inventory Level"],
    "Yes","No"
)

In [24]:
df[["Inventory Level", "Demand Forecast", "Stockout"]].head()

,Inventory Level,Demand Forecast,Stockout
0,231,135.47,No
1,204,144.04,No
2,102,74.02,No
3,469,62.18,No
4,166,9.26,No


### Overstock

In [25]:
df["Overstock"] = np.where(
    df["Inventory Level"] > df["Demand Forecast"],
    "Yes", "No"
)

In [26]:
df[["Inventory Level", "Demand Forecast", "Overstock"]].head()

,Inventory Level,Demand Forecast,Overstock
0,231,135.47,Yes
1,204,144.04,Yes
2,102,74.02,Yes
3,469,62.18,Yes
4,166,9.26,Yes


In [27]:
print(df["Stockout"].value_counts())
print(df["Overstock"].value_counts())

Stockout
No     70515
Yes     2585
Name: count, dtype: int64
Overstock
Yes    70512
No      2588
Name: count, dtype: int64


here majority of products are overstock, but its neccesary to meet the demand. as per retail view inventory stock must be ate least 20% higher than demand, so we will flag those who have 20% higher stock. here 20% is our threshold value

In [37]:
df["Overstock flag"] = np.where(
    df["Inventory Level"] > (df["Demand Forecast"] * 1.20),
    "Yes", "No"
)

In [38]:
df[["Overstock flag"]].head()

,Overstock flag
0,Yes
1,Yes
2,Yes
3,Yes
4,Yes


In [39]:
df["Overstock flag"].value_counts()

Overstock flag
Yes    59037
No     14063
Name: count, dtype: int64

In [40]:
df["Overstock flag"].value_counts(normalize=True)*100

Overstock flag
Yes    80.76197
No     19.23803
Name: proportion, dtype: float64

so nealry the 80% is overstock, meaning more than expected demand threshold. meaning its inventory has at least 20% higher than the forecasted demand

In [41]:
df.to_csv("feature_engineered_inventory.csv", index=False)

print("Feature engineered dataset saved")

Feature engineered dataset saved
